# Laboratorio 7 - Incisos 11 y 12

In [1]:

import pandas as pd
import numpy as np
import pyreadr
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


In [2]:

result = pyreadr.read_r("data/listings.RData")
df = list(result.values())[0]


In [3]:

df['price_clean'] = (
    df['price'].astype(str)
    .replace(r'[\$,]', '', regex=True)
    .replace('', np.nan)
)

df['price_clean'] = pd.to_numeric(df['price_clean'], errors='coerce')
df = df.dropna(subset=['price_clean'])

q1 = df['price_clean'].quantile(0.33)
q2 = df['price_clean'].quantile(0.66)

def categorize(x):
    if x <= q1:
        return 'barata'
    elif x <= q2:
        return 'media'
    else:
        return 'cara'

df['price_category'] = df['price_clean'].apply(categorize)


In [4]:

cols = [
    'accommodates','bathrooms','minimum_nights',
    'availability_30','number_of_reviews',
    'reviews_per_month','review_scores_rating',
    'latitude','room_type'
]

df_model = df[cols + ['price_category']].dropna()


In [5]:

X = df_model.drop(columns=['price_category'])
y = df_model['price_category']

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
